# Comprehensive EDA of Brazilian Human Rights Hotline Complaints

## Step 3: Data Cleaning and Validation

This notebook applies controlled, reproducible corrections to the Brazilian human rights hotline dataset after the Step 2 quality assessment. It preserves an immutable raw layer, records every rule in an audit log, validates the cleaned result, and compares distributions before and after cleaning.

### Cleaning principles

1. Never overwrite the source dataset.
2. Apply only transparent and approved rules.
3. Preserve correct Portuguese accents in display values.
4. Use accent-insensitive keys only for matching.
5. Treat conditional missingness separately from quality defects.
6. Do not remove rows or columns solely because missingness is high.
7. Do not treat repeated `hash` values as duplicates until the dataset grain is confirmed.
8. Export a correction log and validation report with the cleaned data.
9. Re-run all validation rules after cleaning.

> **Research decision:** the numeric codes `-95`, `-97`, `-98`, `-99`, and `-999` are outside the scope of this research. This notebook does not search for, replace, or interpret them.


## 3.1 Imports and configuration

The notebook can read the original source directly or the proposed output from Step 2. By default, it reads the original source so every correction remains reproducible from the raw layer.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 160)

DATA_PATH = Path("data/disque100.csv")
SHEET_NAME = 0
CSV_ENCODING = "utf-8"
CSV_SEPARATOR = None
OUTPUT_DIR = Path("outputs/step_03_data_cleaning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Safety switches. Review before enabling.
REMOVE_EXACT_DUPLICATES = False
EXPORT_CLEAN_DATA = True
EXPORT_FORMAT = "parquet"  # parquet or csv

print("Pandas:", pd.__version__)


## 3.2 Load and fingerprint the immutable source

A SHA-256 fingerprint is generated for the input file. This supports reproducibility and proves which source version was cleaned.


In [ ]:
def load_dataset(path, sheet_name=0, encoding="utf-8", separator=None):
    ext = path.suffix.lower()
    if ext == ".csv":
        try:
            return pd.read_csv(path, sep=separator, engine="python" if separator is None else "c", encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(path, sep=separator, engine="python" if separator is None else "c", encoding="latin-1", low_memory=False)
    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name)
    if ext == ".parquet":
        return pd.read_parquet(path)
    raise ValueError("Supported formats: CSV, XLSX/XLS, and Parquet")


def sha256_file(path, block_size=1024*1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


if DATA_PATH.exists():
    source_fingerprint = sha256_file(DATA_PATH)
    df_raw = load_dataset(DATA_PATH, SHEET_NAME, CSV_ENCODING, CSV_SEPARATOR)
    df_clean = df_raw.copy(deep=True)
    print(f"Loaded {len(df_raw):,} rows and {df_raw.shape[1]:,} columns")
    print("SHA-256:", source_fingerprint)
else:
    df_raw = df_clean = None
    source_fingerprint = None
    print(f"Update DATA_PATH. File not found: {DATA_PATH.resolve()}")


## 3.3 Declare approved cleaning rules

The dictionaries below are the notebook's control center. Generic formatting rules can run immediately. Semantic category mappings should be added only after validation against the official data dictionary or a documented expert decision.


In [ ]:
TEXT_MISSING_TOKENS = {
    "", "-", "--", ".", "..", "...", "N/A", "NA", "NAN", "NULL", "NONE",
    "SEM INFORMACAO", "SEM INFORMACOES", "SEM DADO", "SEM DADOS",
    "NAO INFORMADO", "NAO INFORMADA", "NAO INFORMADOS", "NAO INFORMADAS",
    "NAO DECLARADO", "NAO DECLARADA", "NAO SE APLICA", "NAO APLICAVEL",
    "NAO SABE", "IGNORADO", "IGNORADA", "DESCONHECIDO", "DESCONHECIDA",
    "INDEFINIDO", "INDEFINIDA", "NI", "NSA"
}

UF_CODES = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT",
    "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO",
    "RR", "SC", "SP", "SE", "TO"
}

# Add only mappings approved for this research.
# Keys are matched without case, accent, or spacing differences.
APPROVED_CATEGORY_MAPPINGS = {
    # "denuncia_emergencial": {
    #     "SIM": "SIM",
    #     "NAO": "NÃO",
    # },
    # "genero_da_vitima": {...},
    # "genero_do_suspeito": {...},
    # "raca_cor_da_vitima": {...},
}

# Optional replacements for known municipality or country variants.
APPROVED_VALUE_MAPPINGS = {
    # "pais": {"BR": "BRASIL", "BRAZIL": "BRASIL"},
}

# Columns that may legitimately have high missingness.
CONDITIONAL_COLUMNS = {
    "motivacao", "sl_vitima_naturalidade", "sl_vitima_naturalizado_uf",
    "sl_vitima_naturalizado_municipio", "deficiencia_da_vitima",
    "deficiencia_relacionada_a_doenca_rara", "vitima_preso_a",
    "profissao_da_vitima", "grau_de_instrucao_da_vitima", "religiao_da_vitima",
    "etnia_da_vitima", "faixa_de_renda_da_vitima", "sl_suspeito_naturalidade",
    "sl_suspeito_naturalizado_uf", "sl_suspeito_naturalizado_municipio",
    "deficiencia_do_suspeito", "deficiencia_relacionada_a_doenca_rara_suspeito",
    "suspeito_preso", "profissao_do_suspeito", "grau_de_instrucao_do_suspeito",
    "religiao_do_suspeito", "suspeito_etnia", "faixa_de_renda_do_suspeito",
    "vinculo_orgao_pj_do_suspeito", "sl_suspeito_ramo",
    "suspeito_especificacao_etnia", "vitima_especificacao_etnia"
}


## 3.4 Canonical matching and audit-log helpers

The audit log records rule, column, original value, corrected value, affected count, and execution timestamp. Row identifiers are not written to the summary log to avoid unnecessary disclosure.


In [ ]:
def strip_accents(text):
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))


def comparison_key(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text.strip()).upper()
    return strip_accents(text)


CANONICAL_MISSING_TOKENS = {comparison_key(x) for x in TEXT_MISSING_TOKENS}
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()
correction_log_rows = []


def record_correction(rule, column, original, corrected, count, notes=""):
    correction_log_rows.append({
        "timestamp_utc": RUN_TIMESTAMP,
        "rule": rule,
        "column": column,
        "original_value": original,
        "corrected_value": corrected,
        "affected_cells": int(count),
        "notes": notes,
    })


def summarize_changes(before, after, rule, column, notes=""):
    before_s = before.astype("string").fillna("<NA>")
    after_s = after.astype("string").fillna("<NA>")
    changed = before_s.ne(after_s)
    if not changed.any():
        return 0
    pairs = pd.DataFrame({"before": before_s[changed], "after": after_s[changed]})
    for (old, new), count in pairs.value_counts().items():
        record_correction(rule, column, old, new, count, notes)
    return int(changed.sum())


## 3.5 Standardize column names safely

Column labels are normalized to lower-case snake case. Duplicate names created by normalization are detected and block the process rather than being silently overwritten.


In [ ]:
def normalize_column_name(name):
    text = strip_accents(unicodedata.normalize("NFKC", str(name)))
    text = re.sub(r"[^0-9A-Za-z]+", "_", text.strip().lower()).strip("_")
    return text


if df_clean is not None:
    original_columns = list(df_clean.columns)
    normalized_columns = [normalize_column_name(c) for c in original_columns]
    duplicate_normalized = pd.Series(normalized_columns)[pd.Series(normalized_columns).duplicated(keep=False)]
    if not duplicate_normalized.empty:
        raise ValueError(f"Column normalization would create duplicates: {sorted(duplicate_normalized.unique())}")
    df_clean.columns = normalized_columns
    column_name_map = pd.DataFrame({"original_column": original_columns, "clean_column": normalized_columns})
    column_name_map["changed"] = column_name_map["original_column"] != column_name_map["clean_column"]
    display(column_name_map[column_name_map["changed"]])


## 3.6 Normalize text formatting

This rule applies Unicode NFKC normalization, trims surrounding whitespace, collapses repeated whitespace, and converts categories to uppercase. Accents are retained in the cleaned display values.


In [ ]:
def normalize_display_text(value):
    if pd.isna(value):
        return pd.NA
    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text.strip())
    return text.upper()


if df_clean is not None:
    text_columns = df_clean.select_dtypes(include=["object", "string", "category"]).columns
    formatting_changes = []
    for column in text_columns:
        before = df_clean[column].copy()
        after = before.map(normalize_display_text).astype("string")
        changed = summarize_changes(
            before, after, "normalize_text_format", column,
            "Unicode NFKC; trim; collapse whitespace; uppercase; preserve accents."
        )
        df_clean[column] = after
        formatting_changes.append({"column": column, "changed_cells": changed})
    formatting_summary = pd.DataFrame(formatting_changes).sort_values("changed_cells", ascending=False)
    display(formatting_summary)


## 3.7 Convert approved textual missing tokens to `pd.NA`

Only the declared text vocabulary is converted. The excluded numeric codes remain untouched.


In [ ]:
if df_clean is not None:
    semantic_missing_changes = []
    for column in df_clean.select_dtypes(include=["object", "string", "category"]).columns:
        before = df_clean[column].copy()
        mask = before.notna() & before.map(comparison_key).isin(CANONICAL_MISSING_TOKENS)
        if mask.any():
            for original, count in before[mask].value_counts(dropna=False).items():
                record_correction(
                    "text_missing_to_na", column, original, "<NA>", count,
                    "Approved text-based missing token. Numeric sentinel codes are excluded."
                )
            df_clean.loc[mask, column] = pd.NA
        semantic_missing_changes.append({"column": column, "converted_to_na": int(mask.sum())})
    semantic_missing_summary = pd.DataFrame(semantic_missing_changes).sort_values("converted_to_na", ascending=False)
    display(semantic_missing_summary)


## 3.8 Apply reviewed category mappings

Mappings are applied through canonical keys, allowing variants in case, accents, or spacing to map to one approved Portuguese display value. Unmapped values remain unchanged.


In [ ]:
def apply_mapping(series, mapping):
    canonical_mapping = {comparison_key(k): v for k, v in mapping.items()}
    return series.map(
        lambda x: canonical_mapping.get(comparison_key(x), x) if not pd.isna(x) else pd.NA
    ).astype("string")


if df_clean is not None:
    mapping_summary = []
    combined_mappings = {**APPROVED_VALUE_MAPPINGS, **APPROVED_CATEGORY_MAPPINGS}
    for column, mapping in combined_mappings.items():
        if column not in df_clean.columns:
            mapping_summary.append({"column": column, "status": "column_not_found", "changed_cells": 0})
            continue
        before = df_clean[column].copy()
        after = apply_mapping(before, mapping)
        changed = summarize_changes(before, after, "approved_category_mapping", column, "Mapping approved in notebook configuration.")
        df_clean[column] = after
        mapping_summary.append({"column": column, "status": "applied", "changed_cells": changed})
    mapping_summary = pd.DataFrame(mapping_summary)
    display(mapping_summary)


## 3.9 Correct data types

`data_de_cadastro` is converted to datetime and `sl_quantidade_vitimas` to nullable integer only when values are parseable and integer-valued. Invalid values are reported rather than silently coerced into a supposedly valid result.


In [ ]:
if df_clean is not None:
    type_conversion_report = []

    if "data_de_cadastro" in df_clean.columns:
        before = df_clean["data_de_cadastro"].copy()
        parsed = pd.to_datetime(before, errors="coerce")
        invalid = before.notna() & parsed.isna()
        type_conversion_report.append({
            "column": "data_de_cadastro", "target_dtype": "datetime64[ns]",
            "invalid_values": int(invalid.sum()), "converted_values": int(parsed.notna().sum())
        })
        df_clean["data_de_cadastro"] = parsed

    if "sl_quantidade_vitimas" in df_clean.columns:
        before = df_clean["sl_quantidade_vitimas"].copy()
        numeric = pd.to_numeric(before, errors="coerce")
        invalid = before.notna() & numeric.isna()
        fractional = numeric.notna() & ((numeric % 1) != 0)
        if fractional.any():
            type_conversion_report.append({
                "column": "sl_quantidade_vitimas", "target_dtype": "Int64",
                "invalid_values": int(invalid.sum()), "converted_values": 0,
                "note": f"Conversion blocked: {int(fractional.sum())} fractional values require review."
            })
        else:
            df_clean["sl_quantidade_vitimas"] = numeric.astype("Int64")
            type_conversion_report.append({
                "column": "sl_quantidade_vitimas", "target_dtype": "Int64",
                "invalid_values": int(invalid.sum()), "converted_values": int(numeric.notna().sum()),
                "note": "Nonparseable source values remain missing after coercion and must be reviewed."
            })

    type_conversion_report = pd.DataFrame(type_conversion_report)
    display(type_conversion_report)


## 3.10 Validate UF fields without guessing corrections

Valid UF codes are standardized to uppercase by the formatting rule. Values outside the 27 official codes are quarantined in a validation report and are not guessed or replaced automatically.


In [ ]:
if df_clean is not None:
    uf_columns = [
        c for c in ["uf", "uf_da_vitima", "uf_do_suspeito",
                    "sl_vitima_naturalizado_uf", "sl_suspeito_naturalizado_uf"]
        if c in df_clean.columns
    ]
    uf_invalid_rows = []
    for column in uf_columns:
        invalid = df_clean[column].notna() & ~df_clean[column].isin(UF_CODES)
        for value, count in df_clean.loc[invalid, column].value_counts().items():
            uf_invalid_rows.append({"column": column, "invalid_value": value, "count": int(count)})
    uf_validation = pd.DataFrame(uf_invalid_rows)
    display(uf_validation)


## 3.11 Exact duplicates and repeated identifiers

Exact duplicates can be removed only through the explicit safety switch. Repeated `hash` values are never dropped automatically because they may represent distinct violations or relationships within a complaint.


In [ ]:
if df_clean is not None:
    rows_before_duplicates = len(df_clean)
    exact_duplicate_mask = df_clean.duplicated(keep="first")
    exact_duplicates_found = int(exact_duplicate_mask.sum())

    if REMOVE_EXACT_DUPLICATES and exact_duplicates_found:
        df_clean = df_clean.loc[~exact_duplicate_mask].copy()
        record_correction(
            "remove_exact_duplicate_rows", "<row>", "exact duplicate", "removed",
            exact_duplicates_found, "Enabled by REMOVE_EXACT_DUPLICATES safety switch."
        )

    repeated_hash_rows = 0
    repeated_hash_groups = 0
    if "hash" in df_clean.columns:
        repeated_mask = df_clean["hash"].notna() & df_clean["hash"].duplicated(keep=False)
        repeated_hash_rows = int(repeated_mask.sum())
        repeated_hash_groups = int(df_clean.loc[repeated_mask, "hash"].nunique())

    duplicate_decision = pd.DataFrame({
        "metric": ["Exact duplicates found", "Exact duplicates removed", "Rows in repeated hashes", "Repeated hash groups"],
        "value": [exact_duplicates_found, rows_before_duplicates-len(df_clean), repeated_hash_rows, repeated_hash_groups]
    })
    display(duplicate_decision)


## 3.12 Derive analysis-ready temporal features

These features are deterministic derivations, not corrections. They support later temporal EDA without changing the original registration date.


In [ ]:
if df_clean is not None and "data_de_cadastro" in df_clean.columns:
    df_clean["ano_cadastro"] = df_clean["data_de_cadastro"].dt.year.astype("Int64")
    df_clean["mes_cadastro"] = df_clean["data_de_cadastro"].dt.month.astype("Int64")
    df_clean["ano_mes_cadastro"] = df_clean["data_de_cadastro"].dt.to_period("M").astype("string")
    df_clean["dia_semana_cadastro"] = df_clean["data_de_cadastro"].dt.day_name().astype("string")
    derived_features = ["ano_cadastro", "mes_cadastro", "ano_mes_cadastro", "dia_semana_cadastro"]
    display(df_clean[derived_features].head())


## 3.13 Post-cleaning validation suite

A cleaned dataset is acceptable only if it passes structural, type, formatting, and domain checks. Some findings are warnings rather than failures because they require domain interpretation.


In [ ]:
def count_text_format_issues(data):
    results = []
    for column in data.select_dtypes(include=["object", "string", "category"]).columns:
        s = data[column].astype("string")
        mask = (
            (s.str.match(r"^\s") | s.str.match(r".*\s$")).fillna(False)
            | s.str.contains(r"\s{2,}", regex=True, na=False)
            | s.str.contains(r"[\t\r\n]", regex=True, na=False)
        )
        results.append({"column": column, "issues": int(mask.sum())})
    return pd.DataFrame(results)


if df_clean is not None:
    validation_rows = []
    validation_rows.append({"check": "row_count_not_increased", "status": "PASS" if len(df_clean) <= len(df_raw) else "FAIL", "value": len(df_clean)})
    validation_rows.append({"check": "column_names_unique", "status": "PASS" if df_clean.columns.is_unique else "FAIL", "value": int(df_clean.columns.duplicated().sum())})
    validation_rows.append({"check": "hash_present", "status": "PASS" if "hash" in df_clean.columns else "FAIL", "value": "hash" in df_clean.columns})

    residual_format = count_text_format_issues(df_clean)
    residual_format_count = int(residual_format["issues"].sum())
    validation_rows.append({"check": "no_residual_whitespace_issues", "status": "PASS" if residual_format_count == 0 else "FAIL", "value": residual_format_count})

    residual_semantic = 0
    for column in df_clean.select_dtypes(include=["object", "string", "category"]).columns:
        residual_semantic += int((df_clean[column].notna() & df_clean[column].map(comparison_key).isin(CANONICAL_MISSING_TOKENS)).sum())
    validation_rows.append({"check": "no_residual_text_missing_tokens", "status": "PASS" if residual_semantic == 0 else "FAIL", "value": residual_semantic})

    invalid_uf_count = int(uf_validation["count"].sum()) if not uf_validation.empty else 0
    validation_rows.append({"check": "all_uf_values_valid", "status": "PASS" if invalid_uf_count == 0 else "REVIEW", "value": invalid_uf_count})

    if "data_de_cadastro" in df_clean.columns:
        future_dates = int((df_clean["data_de_cadastro"].notna() & (df_clean["data_de_cadastro"] > pd.Timestamp.now())).sum())
        validation_rows.append({"check": "no_future_registration_dates", "status": "PASS" if future_dates == 0 else "REVIEW", "value": future_dates})

    if "sl_quantidade_vitimas" in df_clean.columns:
        non_positive = int((df_clean["sl_quantidade_vitimas"].notna() & (df_clean["sl_quantidade_vitimas"] <= 0)).sum())
        validation_rows.append({"check": "victim_quantity_positive", "status": "PASS" if non_positive == 0 else "REVIEW", "value": non_positive})

    post_clean_validation = pd.DataFrame(validation_rows)
    display(post_clean_validation)
    display(residual_format[residual_format["issues"] > 0])


## 3.14 Compare before and after cleaning

The comparison detects whether cleaning unexpectedly changes row counts, missingness, or category cardinality. Large reductions in distinct values should correspond to approved normalization rather than accidental information loss.


In [ ]:
if df_clean is not None:
    shared_columns = [c for c in df_clean.columns if c in df_raw.columns]
    comparison_rows = []
    for column in shared_columns:
        comparison_rows.append({
            "column": column,
            "dtype_before": str(df_raw[column].dtype),
            "dtype_after": str(df_clean[column].dtype),
            "missing_before": int(df_raw[column].isna().sum()),
            "missing_after": int(df_clean[column].isna().sum()),
            "unique_before": int(df_raw[column].nunique(dropna=True)),
            "unique_after": int(df_clean[column].nunique(dropna=True)),
        })
    before_after = pd.DataFrame(comparison_rows)
    before_after["missing_change"] = before_after["missing_after"] - before_after["missing_before"]
    before_after["unique_change"] = before_after["unique_after"] - before_after["unique_before"]
    display(before_after.sort_values(["missing_change", "unique_change"], ascending=[False, True]))


### Interactive change charts


In [ ]:
if df_clean is not None:
    missing_plot = before_after[before_after["missing_change"] != 0].sort_values("missing_change")
    if not missing_plot.empty:
        fig = px.bar(
            missing_plot, x="missing_change", y="column", orientation="h",
            color="missing_change", color_continuous_scale="RdBu_r",
            title="Change in missing values after cleaning",
            labels={"missing_change": "Additional missing values after approved conversion"},
            height=max(450, len(missing_plot)*28)
        )
        fig.update_layout(template="plotly_white")
        fig.show()

    unique_plot = before_after[before_after["unique_change"] != 0].copy()
    unique_plot["reduction_in_unique"] = -unique_plot["unique_change"]
    unique_plot = unique_plot.sort_values("reduction_in_unique")
    if not unique_plot.empty:
        fig = px.bar(
            unique_plot, x="reduction_in_unique", y="column", orientation="h",
            title="Reduction in distinct values after cleaning",
            height=max(450, len(unique_plot)*28)
        )
        fig.update_layout(template="plotly_white")
        fig.show()


## 3.15 Distribution stability checks

For priority categorical variables, this section compares proportions before and after cleaning. Cleaning should merge equivalent labels and convert approved missing tokens without materially changing substantive patterns.


In [ ]:
def distribution_table(data, column):
    s = data[column].astype("string").fillna("<MISSING>")
    counts = s.value_counts(dropna=False)
    return pd.DataFrame({"category": counts.index, "count": counts.values, "proportion": counts.values/len(data)})


if df_clean is not None:
    priority_columns = [
        c for c in ["canal_de_atendimento", "denuncia_emergencial", "grupo_vulneravel",
                    "genero_da_vitima", "faixa_etaria_da_vitima", "raca_cor_da_vitima",
                    "genero_do_suspeito", "faixa_etaria_do_suspeito",
                    "relacao_vitima_suspeito", "violacao", "uf"]
        if c in df_clean.columns and c in df_raw.columns
    ]
    stability_rows = []
    for column in priority_columns:
        before = distribution_table(df_raw, column).rename(columns={"count":"count_before", "proportion":"proportion_before"})
        after = distribution_table(df_clean, column).rename(columns={"count":"count_after", "proportion":"proportion_after"})
        merged = before.merge(after, on="category", how="outer").fillna(0)
        total_variation = 0.5 * (merged["proportion_before"] - merged["proportion_after"]).abs().sum()
        stability_rows.append({
            "column": column,
            "total_variation_distance": round(float(total_variation), 6),
            "categories_before": int(df_raw[column].nunique(dropna=True)),
            "categories_after": int(df_clean[column].nunique(dropna=True))
        })
    distribution_stability = pd.DataFrame(stability_rows).sort_values("total_variation_distance", ascending=False)
    display(distribution_stability)


## 3.16 Correction log and cleaning summary


In [ ]:
if df_clean is not None:
    correction_log = pd.DataFrame(correction_log_rows)
    if correction_log.empty:
        correction_log = pd.DataFrame(columns=[
            "timestamp_utc", "rule", "column", "original_value",
            "corrected_value", "affected_cells", "notes"
        ])
    cleaning_summary = correction_log.groupby(["rule", "column"], dropna=False, as_index=False).agg(
        affected_cells=("affected_cells", "sum"),
        distinct_replacements=("original_value", "nunique")
    ).sort_values("affected_cells", ascending=False) if not correction_log.empty else pd.DataFrame()
    display(cleaning_summary)
    display(correction_log.head(100))


## 3.17 Export cleaned data and audit artifacts

The preferred output is Parquet because it preserves nullable integers, datetimes, and strings. CSV remains available for interoperability. The export includes metadata sufficient to reproduce the cleaning run.


In [ ]:
if df_clean is not None:
    run_metadata = {
        "run_timestamp_utc": RUN_TIMESTAMP,
        "source_path": str(DATA_PATH),
        "source_sha256": source_fingerprint,
        "source_rows": int(len(df_raw)),
        "source_columns": int(df_raw.shape[1]),
        "clean_rows": int(len(df_clean)),
        "clean_columns": int(df_clean.shape[1]),
        "remove_exact_duplicates": REMOVE_EXACT_DUPLICATES,
        "numeric_missing_codes_processed": False,
        "export_format": EXPORT_FORMAT,
    }

    column_name_map.to_csv(OUTPUT_DIR / "column_name_map.csv", index=False, encoding="utf-8-sig")
    formatting_summary.to_csv(OUTPUT_DIR / "formatting_summary.csv", index=False, encoding="utf-8-sig")
    semantic_missing_summary.to_csv(OUTPUT_DIR / "semantic_missing_summary.csv", index=False, encoding="utf-8-sig")
    mapping_summary.to_csv(OUTPUT_DIR / "mapping_summary.csv", index=False, encoding="utf-8-sig")
    type_conversion_report.to_csv(OUTPUT_DIR / "type_conversion_report.csv", index=False, encoding="utf-8-sig")
    uf_validation.to_csv(OUTPUT_DIR / "uf_validation.csv", index=False, encoding="utf-8-sig")
    duplicate_decision.to_csv(OUTPUT_DIR / "duplicate_decision.csv", index=False, encoding="utf-8-sig")
    post_clean_validation.to_csv(OUTPUT_DIR / "post_clean_validation.csv", index=False, encoding="utf-8-sig")
    before_after.to_csv(OUTPUT_DIR / "before_after_cleaning.csv", index=False, encoding="utf-8-sig")
    distribution_stability.to_csv(OUTPUT_DIR / "distribution_stability.csv", index=False, encoding="utf-8-sig")
    correction_log.to_csv(OUTPUT_DIR / "correction_log.csv", index=False, encoding="utf-8-sig")
    cleaning_summary.to_csv(OUTPUT_DIR / "cleaning_summary.csv", index=False, encoding="utf-8-sig")
    (OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, ensure_ascii=False, indent=2), encoding="utf-8")

    if EXPORT_CLEAN_DATA:
        if EXPORT_FORMAT.lower() == "parquet":
            try:
                df_clean.to_parquet(OUTPUT_DIR / "disque100_clean.parquet", index=False)
                exported_file = OUTPUT_DIR / "disque100_clean.parquet"
            except ImportError:
                df_clean.to_csv(OUTPUT_DIR / "disque100_clean.csv", index=False, encoding="utf-8-sig")
                exported_file = OUTPUT_DIR / "disque100_clean.csv"
                print("Parquet engine unavailable; exported CSV instead.")
        else:
            df_clean.to_csv(OUTPUT_DIR / "disque100_clean.csv", index=False, encoding="utf-8-sig")
            exported_file = OUTPUT_DIR / "disque100_clean.csv"
        print("Cleaned dataset:", exported_file.resolve())

    print("Audit artifacts:", OUTPUT_DIR.resolve())


## 3.18 Acceptance criteria

The cleaned dataset is ready for substantive EDA when:

- raw data and source fingerprint are preserved
- column names are unique
- residual whitespace problems equal zero
- approved text-missing tokens have been converted consistently
- invalid UF values are resolved or documented
- date and quantity anomalies are resolved or explicitly retained
- repeated `hash` values have an agreed interpretation
- category mappings are approved and documented
- distribution changes are explainable from the correction log
- no validation check has status `FAIL`

## Next step

**Step 4: Univariate Analysis** should use `df_clean` to profile complaint channels, emergency status, vulnerable groups, violation categories, victim and suspect characteristics, geography, and victim quantities with interactive Plotly charts. Counts must follow the dataset-grain decision from Steps 1 to 3.
